In [1]:
!pip install anthropic mlflow requests boto3

import boto3
import mlflow
import requests
import json
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import re
from datasets import load_dataset
from tqdm import tqdm
import mlflow.sklearn 

mlflow.set_experiment("LLM_Retrieval_Accuracy")

Defaulting to user installation because normal site-packages is not writeable


<Experiment: artifact_location='file:///sfs/gpfs/tardis/home/czz7bf/Documents/retrieval_aug_gen/mlruns/232831116680992338', creation_time=1743715221591, experiment_id='232831116680992338', last_update_time=1743715221591, lifecycle_stage='active', name='LLM_Retrieval_Accuracy', tags={}>

In [ ]:
import os
import anthropic

In [8]:
from anthropic import HUMAN_PROMPT, AI_PROMPT
from dotenv import load_dotenv
load_dotenv("env.txt")

In [ ]:
os.environ['ANTHROPIC_API_KEY'] = 
client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))


In [13]:
!pip install boto3 mlflow requests datasets scikit-learn nltk tqdm torch sentence-transformers rank-bm25 anthropic pandas


import os
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util
from rank_bm25 import BM25Okapi
import anthropic
import mlflow
from tqdm import tqdm

class RAGModel:
    def __init__(self, embedding_model="multi-qa-mpnet-base-dot-v1", use_gpu=True, claude_version="claude-3-5-sonnet-20240620"):
        self.device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(embedding_model, device=self.device)
        self.documents = []
        self.embeddings = None
        self.bm25 = None
        self.anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.claude_version = claude_version
    
    def set_claude_version(self, version):
        self.claude_version = version

    def load_documents(self, documents, batch_size=64):
        self.documents = documents
        tokenized_docs = [doc.split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)  

        
        self.embeddings = self.model.encode(documents, convert_to_tensor=True, device=self.device)

    def retrieve(self, query, top_k=10, retrieval_method="hybrid"):
        """Retrieve relevant documents using Embeddings, BM25, or a Hybrid approach."""
        if retrieval_method == "embedding":
            if self.embeddings is None:
                raise RuntimeError("Embeddings have not been loaded. Run load_documents() first.")
            query_embedding = self.model.encode(query, convert_to_tensor=True, device=self.device)
            scores = util.pytorch_cos_sim(query_embedding, self.embeddings)[0]
            top_results = torch.topk(scores, k=top_k)
            return [(self.documents[idx], scores[idx].item()) for idx in top_results.indices]

        elif retrieval_method == "bm25":
            if self.bm25 is None:
                raise RuntimeError("BM25 index has not been initialized. Run load_documents() first.")
            scores = self.bm25.get_scores(query.split())
            top_indices = np.argsort(scores)[-top_k:][::-1]  
            return [(self.documents[i], scores[i]) for i in top_indices]

        elif retrieval_method == "hybrid":
            if self.bm25 is None or self.embeddings is None:
                raise RuntimeError("Both BM25 and embeddings must be initialized for hybrid retrieval.")
            bm25_results = self.retrieve(query, top_k=top_k, retrieval_method="bm25")
            embedding_results = self.retrieve(query, top_k=top_k, retrieval_method="embedding")
            doc_scores = {doc: score * 0.4 for doc, score in bm25_results}
            for doc, score in embedding_results:
                doc_scores[doc] = doc_scores.get(doc, 0) + score * 0.6
            sorted_results = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
            return sorted_results[:top_k]

        else:
            raise ValueError("Invalid retrieval method. Choose 'embedding', 'bm25', or 'hybrid'.")

    def generate_answer(self, query, context):
        prompt = f"""
        You have been tasked with answering the following query:
        <query>
        {query}
        </query>
        Based on the following context:
        <context>
        {context}
        </context>
        Provide a concise and accurate response.
        """
        response = self.anthropic_client.messages.create(
            model=self.claude_version,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.content[0].text


def evaluate_retrieval(model, queries, labels, top_k=10, retrieval_method="hybrid"):
    precision_scores = []
    recall_scores = []
    f1_scores = []
    mrr_scores = []
    epsilon = 1e-10 

    for i, query in enumerate(queries):  
        relevant_label = labels[i]  
        retrieved_docs = model.retrieve(query, top_k=top_k, retrieval_method=retrieval_method)

        label_embedding = model.model.encode(relevant_label, convert_to_tensor=True)

        relevance = []
        for doc, _ in retrieved_docs:
            doc_embedding = model.model.encode(doc, convert_to_tensor=True)
            similarity = util.pytorch_cos_sim(label_embedding, doc_embedding)[0].item()
            relevance.append(1 if similarity >= 0.3 else 0) 

        precision = sum(relevance) / top_k if top_k > 0 else 0
        precision_scores.append(precision)

        relevant_count = sum(relevance)  
        total_relevant_in_corpus = max(1, sum(1 for lbl in labels if lbl.strip().lower() == relevant_label.strip().lower()))
        recall = min(relevant_count / total_relevant_in_corpus, 1.0) if total_relevant_in_corpus > 0 else 0
        recall_scores.append(recall)

        
        f1_score = (2 * precision * recall) / (precision + recall + epsilon)  
        f1_scores.append(f1_score)

        if 1 in relevance:
            mrr_scores.append(1 / (relevance.index(1) + 1))
        else:
            mrr_scores.append(0)

    avg_precision = np.mean(precision_scores) if precision_scores else 0
    avg_recall = np.mean(recall_scores) if recall_scores else 0
    avg_f1 = np.mean(f1_scores) if f1_scores else 0
    avg_mrr = np.mean(mrr_scores) if mrr_scores else 0

    return {
        "Precision_K": avg_precision,  
        "Recall_K": avg_recall,      
        "F1_Score": avg_f1,          
        "MRR": avg_mrr
    }

mlflow.set_experiment("RAG_Model_Evaluation")

squad_dataset = load_dataset("squad", split="validation")
queries = squad_dataset["question"]  
ground_truths_list = squad_dataset["answers"]  
contexts = [ctx for context_group in squad_dataset["context"] for ctx in context_group]  


rag_model = RAGModel(claude_version="claude-3-5-sonnet-20240620")
rag_model.load_documents(documents)

with mlflow.start_run(run_name="Claude_3.5_Sonnet_Evaluation"):
    metrics = evaluate_retrieval(rag_model, queries[:100], ground_truths_list[:100], contexts, top_k=10, retrieval_method="hybrid")

    mlflow.log_param("model_version", "claude-3-5-sonnet-20240620")
    mlflow.log_metric("Precision_K", metrics["Precision_K"])  
    mlflow.log_metric("Recall_K", metrics["Recall_K"])        
    mlflow.log_metric("F1_Score", metrics["F1_Score"])        
    mlflow.log_metric("MRR", metrics["MRR"])

    print(f"Metrics for Claude 3.5 Sonnet: {metrics}")

squad_results = pd.DataFrame.from_dict({"Claude 3.5 Sonnet": metrics}, orient="index")
print(squad_results)

# Save to CSV
squad_results.to_csv("retrieval_metrics_single_version.csv", index=True)
print("Results saved to retrieval_metrics_single_version.csv")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Defaulting to user installation because normal site-packages is not writeable


/home/czz7bf/.local/lib/python3.11/site-packages/datasets/load.py:1491: FutureWarning: The repository for trec contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/trec
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Metrics for Claude 3.5 Sonnet: {'Precision_K': 0.93, 'Recall_K': 1.0, 'F1_Score': 0.9607155142261309, 'MRR': 0.8833333333333332}
                   Precision_K  Recall_K  F1_Score       MRR
Claude 3.5 Sonnet         0.93       1.0  0.960716  0.883333
Results saved to retrieval_metrics_single_version.csv


In [12]:
class RAGModel:
    def __init__(self, embedding_model="multi-qa-mpnet-base-dot-v1", use_gpu=True, claude_version="claude-3-5-sonnet-20240620"):
        self.device = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"
        self.model = SentenceTransformer(embedding_model, device=self.device)
        self.documents = []
        self.embeddings = None
        self.bm25 = None
        self.anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.claude_version = claude_version
    
    def set_claude_version(self, version):
        self.claude_version = version

    def load_documents(self, documents, batch_size=64):
        self.documents = documents
        tokenized_docs = [doc.split() for doc in documents]
        self.bm25 = BM25Okapi(tokenized_docs)  

        self.embeddings = self.model.encode(documents, convert_to_tensor=True, device=self.device)

    def retrieve(self, query, top_k=10, retrieval_method="hybrid"):
        """Retrieve relevant documents using Embeddings, BM25, or a Hybrid approach."""
        if retrieval_method == "embedding":
            if self.embeddings is None:
                raise RuntimeError("Embeddings have not been loaded. Run load_documents() first.")
            query_embedding = self.model.encode(query, convert_to_tensor=True, device=self.device)
            scores = util.pytorch_cos_sim(query_embedding, self.embeddings)[0]
            top_results = torch.topk(scores, k=top_k)
            return [(self.documents[idx], scores[idx].item()) for idx in top_results.indices]

        elif retrieval_method == "bm25":
            if self.bm25 is None:
                raise RuntimeError("BM25 index has not been initialized. Run load_documents() first.")
            scores = self.bm25.get_scores(query.split())
            top_indices = np.argsort(scores)[-top_k:][::-1]  
            return [(self.documents[i], scores[i]) for i in top_indices]

        elif retrieval_method == "hybrid":
            if self.bm25 is None or self.embeddings is None:
                raise RuntimeError("Both BM25 and embeddings must be initialized for hybrid retrieval.")
            bm25_results = self.retrieve(query, top_k=top_k, retrieval_method="bm25")
            embedding_results = self.retrieve(query, top_k=top_k, retrieval_method="embedding")
            doc_scores = {doc: score * 0.4 for doc, score in bm25_results}
            for doc, score in embedding_results:
                doc_scores[doc] = doc_scores.get(doc, 0) + score * 0.6
            sorted_results = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
            return sorted_results[:top_k]

        else:
            raise ValueError("Invalid retrieval method. Choose 'embedding', 'bm25', or 'hybrid'.")

    def generate_answer(self, query, context):
        prompt = f"""
        You have been tasked with answering the following query:
        <query>
        {query}
        </query>
        Based on the following context:
        <context>
        {context}
        </context>
        Provide a concise and accurate response.
        """
        response = self.anthropic_client.messages.create(
            model=self.claude_version,
            max_tokens=1024,
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        return response.content[0].text


def evaluate_retrieval(model, queries, ground_truths_list, contexts, top_k=10, retrieval_method="hybrid"):
    precision_scores = []
    recall_scores = []
    f1_scores = []
    mrr_scores = []
    epsilon = 1e-10 

    for i, query in enumerate(queries):  
        ground_truths = ground_truths_list[i] 
        retrieved_docs = model.retrieve(query, top_k=top_k, retrieval_method=retrieval_method)

        relevance = []
        for doc, _ in retrieved_docs:
            relevance.append(1 if any(gt.strip().lower() in doc.strip().lower() for gt in ground_truths) else 0)

        precision = sum(relevance) / top_k if top_k > 0 else 0
        precision_scores.append(precision)

        relevant_count = sum(relevance)  
        total_relevant_in_corpus = max(1, sum(1 for gt in ground_truths if any(gt.strip().lower() in ctx.strip().lower() for ctx in contexts)))
        recall = min(relevant_count / total_relevant_in_corpus, 1.0) if total_relevant_in_corpus > 0 else 0
        recall_scores.append(recall)

        f1_score_value = 2 * precision * recall / (precision + recall + epsilon)
        f1_scores.append(f1_score_value)

        if 1 in relevance:
            mrr_scores.append(1 / (relevance.index(1) + 1))
        else:
            mrr_scores.append(0)

    avg_precision = np.mean(precision_scores) if precision_scores else 0
    avg_recall = np.mean(recall_scores) if recall_scores else 0
    avg_f1 = np.mean(f1_scores) if f1_scores else 0
    avg_mrr = np.mean(mrr_scores) if mrr_scores else 0

    return {
        "Precision_K": avg_precision,
        "Recall_K": avg_recall,
        "F1_Score": avg_f1,
        "MRR": avg_mrr
    }

fiqa_dataset = load_dataset("explodinggradients/fiqa", split="baseline")

queries = fiqa_dataset["question"]  
ground_truths_list = fiqa_dataset["ground_truths"]  
contexts = [ctx for context_group in fiqa_dataset["contexts"] for ctx in context_group]  

rag_model = RAGModel(claude_version="claude-3-5-sonnet-20240620")
rag_model.load_documents(contexts)

mlflow.set_experiment("RAG_Model_Evaluation_FiQA")

with mlflow.start_run(run_name="Claude_3.5_Sonnet_Evaluation_FiQA"):
    
    metrics = evaluate_retrieval(rag_model, queries[:100], ground_truths_list[:100], contexts, top_k=10, retrieval_method="hybrid")


    mlflow.log_param("model_version", "claude-3-5-sonnet-20240620")
    mlflow.log_metric("Precision_K", metrics["Precision_K"])
    mlflow.log_metric("Recall_K", metrics["Recall_K"])
    mlflow.log_metric("F1_Score", metrics["F1_Score"])
    mlflow.log_metric("MRR", metrics["MRR"])

    print(f"Metrics for Claude 3.5 Sonnet on FiQA Dataset: {metrics}")


fiqa_results = pd.DataFrame.from_dict({"Claude 3.5 Sonnet on FiQA": metrics}, orient="index")
print(fiqa_results)

fiqa_results.to_csv("fiqa_retrieval_metrics.csv", index=True)
print("Results saved to fiqa_retrieval_metrics.csv")

/home/czz7bf/.local/lib/python3.11/site-packages/datasets/load.py:1491: FutureWarning: The repository for explodinggradients/fiqa contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/explodinggradients/fiqa
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Metrics for Claude 3.5 Sonnet on FiQA Dataset: {'Precision_K': 0.04333333333333334, 'Recall_K': 0.43333333333333335, 'F1_Score': 0.07878787878071623, 'MRR': 0.4066666666666666}
                           Precision_K  Recall_K  F1_Score       MRR
Claude 3.5 Sonnet on FiQA     0.043333  0.433333  0.078788  0.406667
Results saved to fiqa_retrieval_metrics.csv


In [20]:
comparison_table = pd.concat([squad_results, fiqa_results], axis=0)
comparison_table.reset_index(drop=True, inplace=True)
print(comparison_table)
comparison_table.to_csv("MLFlow_dataset_comparison_results.csv", index=False)
print("Comparison table saved to 'MLFlow_dataset_comparison_results.csv'")

   Precision_K  Recall_K  F1_Score       MRR
0     0.930000  1.000000  0.960716  0.883333
1     0.043333  0.433333  0.078788  0.406667
Comparison table saved to 'MLFlow_dataset_comparison_results.csv'
